In [2]:
import os

pdf = r'C:\Users\Windows\Desktop\projects\genai\rag_chatbot\Kokum_Network_Programming_Manual_0_29_0.pdf'

if os.path.exists(pdf):
    print("Uploaded:", os.path.basename(pdf))
else:
    print("PDF not found")

Uploaded: Kokum_Network_Programming_Manual_0_29_0.pdf


In [23]:
from langchain_community.document_loaders import PyMuPDFLoader
loader=PyMuPDFLoader(pdf)
documents=loader.load()
print("document loaded",len(documents))

document loaded 67


In [24]:
print(documents[0]) #page 1 content

page_content='KOKUM
NETWORK
PROGRAMMING
User Manual & API Reference
Remote databases. Live connections.
HTTP clients. Services. Shared development.
Kokum 0.29.0 |
First networking edition • 9 September 2026
A practical guide with small, source-aligned examples
TigerDB Solutions Private Limited
Kokum programming language created by Anil Jagtap
kokum.dev' metadata={'producer': 'ONLYOFFICE/9.4.0.129', 'creator': 'ONLYOFFICE/9.4.0.129', 'creationdate': '2026-09-09T13:11:54+00:00', 'source': 'C:\\Users\\Windows\\Desktop\\projects\\genai\\rag_chatbot\\Kokum_Network_Programming_Manual_0_29_0.pdf', 'file_path': 'C:\\Users\\Windows\\Desktop\\projects\\genai\\rag_chatbot\\Kokum_Network_Programming_Manual_0_29_0.pdf', 'total_pages': 67, 'format': 'PDF 1.7', 'title': 'Kokum Network Programming — User Manual & API Reference', 'author': 'TigerDB Solutions Private Limited', 'subject': 'Networking in Kokum 0.29.0, Phase 4.17.7', 'keywords': 'Kokum, networking, Remote FlatDB, WebSocket, URLGET, URLPOST

In [25]:
#chunking 
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)
chunks=text_splitter.split_documents(documents)
print('documents has been chunked and the length is ',len(chunks))

documents has been chunked and the length is  289


In [26]:
print(chunks[15].page_content)

LAN IDE is not running the compiler or opening database sockets itself.
The HTTP function names are URLGET and URLPOST. HTTPGET, HTTPPOST, URLGETASYNC and URLPOSTASYNC
are not native aliases in this release. Async behavior is composed with Kokum’s existing task language.
SCOPE No public raw TCP/UDP, SMTP/IMAP, FTP/SFTP, HTTP/2/3, SSE, generic WebSocket server or automatic proxy-


In [27]:
print(chunks[16].page_content)

discovery API is documented in this source line. A CHANNEL is an in-process synchronization object, not a network socket.
Source basis: [K02], [K04]–[K13].


In [28]:
#generating the embeddings 
import faiss
import os 
from sentence_transformers import SentenceTransformer
import numpy as np
model=SentenceTransformer('all-MiniLM-L6-v2')
embeddings=model.encode([chunk.page_content for chunk in chunks])
dimension=embeddings.shape[1]
index=faiss.IndexFlatL2(dimension) #it make use of euclidean distance and not approximate nearest neighbour 
index.add(np.array(embeddings)) #adding the embddings into the index
print("total vectors in index:",index.ntotal)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11805.90it/s]


total vectors in index: 289


In [29]:
print(dimension)

384


In [43]:
#query embedding 
query=input("enter your query:")
query_embedding=model.encode([query])
print(query,query_embedding)


what is networking in kokum  [[ 1.09679569e-02 -5.23543954e-02 -4.09976915e-02  4.42178063e-02
   6.90703169e-02  6.44045696e-02 -1.05544906e-02  2.01578457e-02
  -1.41787780e-02  3.14616151e-02 -2.36671418e-02  4.82526608e-02
   3.91671434e-03 -4.34044562e-02  1.85603350e-02  5.36882617e-02
   6.91712275e-02 -3.13764298e-03  1.97016541e-02 -6.32810295e-02
  -5.41021526e-02 -3.21915634e-02  4.86015305e-02 -4.99129929e-02
  -1.71267744e-02 -4.21926342e-02  3.38380300e-02  5.15943728e-02
   1.30725056e-02 -3.02070398e-02 -4.26001335e-03  8.70517939e-02
  -6.05185106e-02  1.07427072e-02 -1.05036400e-01  1.78463776e-02
  -3.40027735e-02 -5.97956590e-02 -1.24779399e-02 -2.51874700e-02
   5.74715324e-02 -1.09693529e-02  7.71435164e-03 -8.19730312e-02
   3.55556048e-02  3.98399942e-02 -8.90399069e-02 -4.32484262e-02
  -1.97423715e-02 -4.37074304e-02 -3.19630913e-02  5.94824972e-03
   2.01521479e-02  8.94034654e-02  8.27985778e-02  4.78061549e-02
  -1.82127561e-02  2.33547725e-02 -9.39254882e-

In [44]:
D,I =index.search(np.array(query_embedding),k=5)
retrieved_chunks=[chunks[i] for i in I[0]]
print("top retrieved chunks are : \n")
for chunk in retrieved_chunks:
    print(chunk.page_content)
    print("page:",chunk.metadata["page"])
    print("***********")

top retrieved chunks are : 

KOKUM / NETWORK PROGRAMMING
USER MANUAL & API REFERENCE
TigerDB Solutions Private Limited | kokum.dev
52
11 • BROWSER BRIDGES AND DEPLOYMENT
Keep the UI transport separate from application
networking
A Web GUI uses a tokenized loopback host and a form bridge. Browser events are validated, dispatched to
exported Kokum slots and converted into patches. An outgoing URL request or WebSocket feed is a different
connection owned by KokumVM.
Connection
Purpose
Boundary
Browser →local Web GUI
page: 51
***********
KOKUM / NETWORK PROGRAMMING
USER MANUAL & API REFERENCE
TigerDB Solutions Private Limited | kokum.dev
6
01 • NETWORK FEATURE MAP
Where the connection actually runs
Layer
What it owns
What it must not own
Browser frontend
Visible controls, user input, selected result data
Database passwords, private keys or native
connection handles
KokumVM owner
session
DATABASE/WEBSOCKET values, GUI state, task
results
Another session’s mutable native objects
Task worker


In [45]:
# Reranking
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

pairs = [[query, chunk.page_content] for chunk in retrieved_chunks]

scores = reranker.predict(pairs)

ranked_chunks = sorted(
    zip(scores, retrieved_chunks),
    key=lambda x: x[0],
    reverse=True
)

print("After reranking:\n")

for score, chunk in ranked_chunks:
    print("Score:", score)
    print(chunk.page_content)
    print("Page number:", chunk.metadata["page"])
    print("---------------------------------------")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3356.57it/s]


After reranking:

Score: 4.699075
KOKUM / NETWORK PROGRAMMING
USER MANUAL & API REFERENCE
TigerDB Solutions Private Limited | kokum.dev
52
11 • BROWSER BRIDGES AND DEPLOYMENT
Keep the UI transport separate from application
networking
A Web GUI uses a tokenized loopback host and a form bridge. Browser events are validated, dispatched to
exported Kokum slots and converted into patches. An outgoing URL request or WebSocket feed is a different
connection owned by KokumVM.
Connection
Purpose
Boundary
Browser →local Web GUI
Page number: 51
---------------------------------------
Score: 4.618784
KOKUM
NETWORK
PROGRAMMING
User Manual & API Reference
Remote databases. Live connections.
HTTP clients. Services. Shared development.
Kokum 0.29.0 |
First networking edition • 9 September 2026
A practical guide with small, source-aligned examples
TigerDB Solutions Private Limited
Kokum programming language created by Anil Jagtap
kokum.dev
Page number: 0
---------------------------------------
Score: 2

In [46]:
#combined reranked chunks with the context 
top_chunks=[chunk.page_content for _, chunk in ranked_chunks[:3]]
context="\n\n".join(top_chunks)
print(context)

KOKUM / NETWORK PROGRAMMING
USER MANUAL & API REFERENCE
TigerDB Solutions Private Limited | kokum.dev
52
11 • BROWSER BRIDGES AND DEPLOYMENT
Keep the UI transport separate from application
networking
A Web GUI uses a tokenized loopback host and a form bridge. Browser events are validated, dispatched to
exported Kokum slots and converted into patches. An outgoing URL request or WebSocket feed is a different
connection owned by KokumVM.
Connection
Purpose
Boundary
Browser →local Web GUI

KOKUM
NETWORK
PROGRAMMING
User Manual & API Reference
Remote databases. Live connections.
HTTP clients. Services. Shared development.
Kokum 0.29.0 |
First networking edition • 9 September 2026
A practical guide with small, source-aligned examples
TigerDB Solutions Private Limited
Kokum programming language created by Anil Jagtap
kokum.dev

KOKUM / NETWORK PROGRAMMING
USER MANUAL & API REFERENCE
TigerDB Solutions Private Limited | kokum.dev
6
01 • NETWORK FEATURE MAP
Where the connection actually runs
Lay

In [48]:
#generate answer using llm 
from dotenv import load_dotenv
import os

load_dotenv()
from google import genai
client=genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
prompt = f"""
Answer the question using the context below.

Context:
{context}

Question:
{query}

Provide a concise answer.
If you don't find the answer in the context, do not guess.
Say: "The information is not found in the document."
"""
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt
)

print(response.text)


Based on the provided context, the document refers to networking in Kokum as "Kokum Network Programming," which includes remote databases, live connections, HTTP clients, services, and shared development (featured in Kokum 0.29.0, the first networking edition).
